# 실습 [20-1] 기본 순환 신경망(SimpleRNN)

> **수정본 안내 (2026-11 우병오)** — 원본은 모델 정의 셀 하나뿐이며 현재 Colab(TensorFlow 2.16+ / Keras 3)에서는 `Embedding`의 입력 크기가 정해지지 않아 `model.summary()`의 파라미터 수가 0(unbuilt)으로 표시된다. `layers.Input(shape=(None,))`을 추가해 파라미터 수가 나오도록 고치고, 임의의 입력을 넣어 각 층의 출력 모양을 확인하는 셀과 파라미터 수 계산 셀을 덧붙였다.

구조: Embedding(1000, 64) → SimpleRNN(128) → Dense(10, softmax)

In [ ]:
"""
어휘 사이즈는 1000, 은닉 벡터 사이즈는 64로 설정
RNN 내부 유닛은 128개, 출력 클래스의 개수는 10개로 설정
"""
import tensorflow as tf
from tensorflow.keras import layers

model = tf.keras.Sequential()
model.add(layers.Input(shape=(None,), dtype='int32'))   # [수정] 입력을 명시해야 Keras 3에서 summary()에 파라미터 수가 표시됨
model.add(layers.Embedding(input_dim=1000, output_dim=64))
model.add(layers.SimpleRNN(128))
model.add(layers.Dense(10, activation='softmax'))
model.summary()

### 출력 모양 확인

In [ ]:
import numpy as np
# 임의의 문장 배치(문장 2개, 길이 7, 단어 번호 0~999)를 넣어 각 층의 출력 모양 확인
x = np.random.randint(0, 1000, size=(2, 7))
h = x
for layer in model.layers:
    h = layer(h)
    print(f"{layer.name:15s} → {tuple(h.shape)}")
print("클래스별 확률 합:", np.round(np.sum(h.numpy(), axis=-1), 3))

### 파라미터 수 직접 계산

In [ ]:
# SimpleRNN 파라미터 = 입력→은닉 가중치(64×128) + 은닉→은닉 가중치(128×128) + 편향(128)
print("SimpleRNN:", 64*128 + 128*128 + 128)          # = 24,704
# model.summary()의 Param # 열과 비교해 보자